In [1]:
#@title Setup (4mins)

%pip install torch
%pip install pyrosetta-installer
!python -c 'import pyrosetta_installer; pyrosetta_installer.install_pyrosetta()'
%pip install biopython
%pip install tqdm
%pip install dm-tree
%pip install biotite
%pip install pyyaml
#!pip install easydict
#!pip install modelcif

!git clone https://github.com/nedru004/MPNN_affinity_maturation.git
#working_dir = "/Workspace/Users/david.nedrud@bio-techne.com/MPNN_affinity_maturation"
working_dir = "/content/MPNN_affinity_maturation"

#!git clone https://gitlab.com/mjslee0921/flowpacker
#%pip install --use-deprecated=legacy-resolver -r flowpacker/requirements.txt

#!git clone https://github.com/cytokineking/ProtenixScore
#%cd $working_dir/MPNN_affinity_maturation/ProtenixScore
#!./install_protenixscore.sh --no-conda

Installing PyRosetta:
 os: ubuntu
 type: Release
 Rosetta C++ extras: 
 mirror: https://west.rosettacommons.org/pyrosetta/release/release
 extra packages: numpy

PyRosetta wheel url: https://:@west.rosettacommons.org/pyrosetta/release/release/PyRosetta4.Release.python312.ubuntu.wheel/pyrosetta-2026.6+release.e5a76a2dbd-cp312-cp312-linux_x86_64.whl
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 GB 770.9 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 GB 770.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 87.3 MB/s eta 0:00:00
   ━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/57.2 MB 162.1 MB/s eta 0:00:01Collecting biotraj<2.0,>=1.0 (from biotite)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 108.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 

In [13]:
#@title 1. MPNN Squence Design
import subprocess
import sys
import json # Added for JSON handling
import os   # Added for path operations

mpnn_script_path = f"{working_dir}/ProteinMPNN/protein_mpnn_run.py" # Renamed 'script' to avoid conflict
folder_with_pdbs = f"{working_dir}/test/"
path_for_parsed_chains = f"{working_dir}/test/parsed_chains.jsonl"
path_for_assigned_chains = f"{working_dir}/test/assigned_chains.jsonl"
path_for_fixed_positions = f"{working_dir}/test/fixed_positions.jsonl" # Ensuring it is jsonl as per MPNN and code definition
out_dir = f"{working_dir}/test/mpnn_out/"
bias_AA_file = f"{working_dir}/bias_AA.jsonl"
chains_to_design ="A"
num_seq_per_target=8
sampling_temp =0.5
seed = 37
fixed_positions="1 2 3 4 5" #fixing/not designing residues 1 2 3...25 in chain A and residues 10 11 12...40 in chain C

!python $working_dir/ProteinMPNN/helper_scripts/parse_multiple_chains.py --input_path=$folder_with_pdbs --output_path=$path_for_parsed_chains

!python $working_dir/ProteinMPNN/helper_scripts/assign_fixed_chains.py --input_path=$path_for_parsed_chains --output_path=$path_for_assigned_chains --chain_list "$chains_to_design"

!python $working_dir/ProteinMPNN/helper_scripts/make_fixed_positions_dict.py --input_path=$path_for_parsed_chains --output_path=$path_for_fixed_positions --chain_list "$chains_to_design" --position_list "$fixed_positions"

# Run actual protein_mpnn_run script
args_for_first_mpnn_run = [
    sys.executable, mpnn_script_path,
    "--jsonl_path", path_for_parsed_chains,
    "--chain_id_jsonl", path_for_assigned_chains,
    "--fixed_positions_jsonl", path_for_fixed_positions,
    "--out_folder", out_dir,
    "--num_seq_per_target", str(num_seq_per_target),
    "--sampling_temp", str(sampling_temp),
    "--seed", str(seed),
    "--batch_size", str(num_seq_per_target),
    "--omit_AAs", "C"]

subprocess.run(args_for_first_mpnn_run)

print(" ".join(args_for_first_mpnn_run))


# MPNN with AA bias
# Ensure correct script path and flag for fixed positions.
args_for_second_mpnn_run = [
    sys.executable, mpnn_script_path,
    "--jsonl_path", path_for_parsed_chains,
    "--chain_id_jsonl", path_for_assigned_chains,
    "--fixed_positions_jsonl", path_for_fixed_positions,
    "--out_folder", out_dir,
    "--num_seq_per_target", str(num_seq_per_target),
    "--sampling_temp", str(sampling_temp),
    "--seed", str(seed),
    "--batch_size", str(num_seq_per_target),
    "--omit_AAs", "C",
    "--bias_AA_jsonl", bias_AA_file]

subprocess.run(args_for_second_mpnn_run)

Traceback (most recent call last):
  File "/content/MPNN_affinity_maturation/ProteinMPNN/helper_scripts/make_fixed_positions_dict.py", line 75, in <module>
    main(args)
  File "/content/MPNN_affinity_maturation/ProteinMPNN/helper_scripts/make_fixed_positions_dict.py", line 56, in main
    fixed_dict = make_fixed_positions_dict(
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/MPNN_affinity_maturation/ProteinMPNN/helper_scripts/make_fixed_positions_dict.py", line 8, in make_fixed_positions_dict
    raise ValueError("Expected a CSV file as --position_list")
ValueError: Expected a CSV file as --position_list
/usr/bin/python3 /content/MPNN_affinity_maturation/ProteinMPNN/protein_mpnn_run.py --jsonl_path /content/MPNN_affinity_maturation/test/parsed_chains.jsonl --chain_id_jsonl /content/MPNN_affinity_maturation/test/assigned_chains.jsonl --fixed_positions_jsonl /content/MPNN_affinity_maturation/test/fixed_positions.jsonl --out_folder /content/MPNN_affinity_maturation/test/mpn

CompletedProcess(args=['/usr/bin/python3', '/content/MPNN_affinity_maturation/ProteinMPNN/protein_mpnn_run.py', '--jsonl_path', '/content/MPNN_affinity_maturation/test/parsed_chains.jsonl', '--chain_id_jsonl', '/content/MPNN_affinity_maturation/test/assigned_chains.jsonl', '--fixed_positions_jsonl', '/content/MPNN_affinity_maturation/test/fixed_positions.jsonl', '--out_folder', '/content/MPNN_affinity_maturation/test/mpnn_out/', '--num_seq_per_target', '8', '--sampling_temp', '0.5', '--seed', '37', '--batch_size', '8', '--omit_AAs', 'C', '--bias_AA_jsonl', '/content/MPNN_affinity_maturation/bias_AA.jsonl'], returncode=1)

In [ ]:
#@title 2. Run flowpacker
#%cd /Workspace/Users/david.nedrud@bio-techne.com/MPNN_affinity_maturation/flowpacker/
#!python $working_dir/sampler_pdb_colab.py base --pdb_dir $working_dir/test/ --save_dir $working_dir/test/out/

%cd flowpacker/
!python $working_dir/sampler_pdb_colab.py base --pdb_dir /Workspace/Users/david.nedrud/Documents/GitHub/MPNN_affinity_maturation/test/mpnn_out --save_dir /Workspace/Users/david.nedrud/Documents/GitHub/MPNN_affinity_maturation/test/out/


/opt/miniconda3/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/Users/david.nedrud/Documents/GitHub/MPNN_affinity_maturation/flowpacker


In [ ]:
#@title 3. Score and filter binders for pTM and ipTM
# boltz score?
from utilities import filter_scores

# ProtenixScore
!python -m protenixscore score --input ./test/flowpacker_out/ --output ./test/ --recursive

# Filter scores
filter_scores("summary.csv", "test/filtered_pdb", threshold_pTM = 0.5, threshold_ipTM = 0.5)

/local_disk0/.ephemeral_nfs/envs/pythonEnv-16c7f3e3-9308-473d-b68f-561c46f53da4/bin/python: No module named protenixscore
python: can't open file '/Workspace/Users/david.nedrud@bio-techne.com/MPNN_affinity_maturation/filter_scores.py': [Errno 2] No such file or directory


In [16]:
#@title 4. Rosetta interaction energy

#pdb_folder = "test"
pdb_file = f"{working_dir}/test/4dn4_ccl2_8_4.pdb"
interface_dist = 10
energy_filter = -5

!python $working_dir/per_residue_energy_pyrosetta.py --pdb $pdb_file --binder_id A --target_id M --interface_dist 10 --output_dir test/

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python312.ubuntu 2026.06+release.e5a76a2dbd7e42e1271d0af460926c86e29e59d1 2026-02-02T18:21:37] retrieved from: http://www.pyrosetta.org
core.init: Checking for fconfig files in pwd and ./rosetta/flags 
core.init: Rosetta version: PyRosetta4.Release.python312.ubuntu r425 2026.06+release.e5a76a2dbd e

In [ ]:
#@title 5. Concatenate key residues

from utilities import process_pdb_mutation_and_renumber
# merge key residues into one pdb file

# Extract Sequences
!python demo_scripts/pdb2fa.py test/af3score/filtered_links csv test/af3score/filtered_links.csv

process_pdb_mutation_and_renumber("test/fixed_residue.csv", "test/merge_motif_pdb/",
    renumber_chain='M', start_index=9) # The residue numbering of the target chain is consistent with the input target file